# 01 - TF-IDF Retrieval from Scratch

**No API key. No ML library. Just Python and math.**

Vectorless RAG means retrieving relevant text without building a vector
embedding. TF-IDF (Term Frequency - Inverse Document Frequency) is one of
the oldest and most reliable ways to score how well a document matches a query
using only word counts.

This notebook builds a complete TF-IDF retrieval system from scratch so you
can see exactly what is happening at each step.

### What you will learn

| Step | Concept |
|------|---------|
| 1 | Split documents into searchable chunks |
| 2 | Compute Term Frequency (TF) for each chunk |
| 3 | Compute Inverse Document Frequency (IDF) across all chunks |
| 4 | Combine TF and IDF to score chunks against a query |
| 5 | Return the top-k most relevant chunks |

> No internet connection or API key required.

## 1. Load the Knowledge Base

In [ ]:
import math
import re
from pathlib import Path
from collections import Counter

# Load all four text files into a list of (source, full_text) tuples
DATA_DIR = Path("../data")

docs = []
for filepath in sorted(DATA_DIR.glob("*.txt")):
    text = filepath.read_text()
    docs.append({"source": filepath.name, "text": text})
    print(f"  Loaded: {filepath.name}  ({len(text)} characters)")

print(f"\nTotal documents: {len(docs)}")

## 2. Split Documents into Chunks

TF-IDF works on chunks, not whole documents. Smaller chunks give more precise
retrieval because each chunk covers a narrower topic.

We split on paragraph boundaries (two newlines) and skip any chunk that is
shorter than 50 characters.

In [ ]:
def split_into_chunks(text, source, min_length=50):
    """Split a document into paragraph-sized chunks."""
    raw_chunks = text.split("\n\n")
    chunks = []
    for chunk in raw_chunks:
        cleaned = chunk.strip()
        if len(cleaned) >= min_length:
            chunks.append({"source": source, "text": cleaned})
    return chunks


all_chunks = []
for doc in docs:
    chunks = split_into_chunks(doc["text"], doc["source"])
    all_chunks.extend(chunks)
    print(f"  {doc['source']:35s}  {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

## 3. Tokenize Text

Tokenization breaks text into individual words (tokens). We lowercase and
remove punctuation so that 'River' and 'river' are treated as the same word.

In [ ]:
def tokenize(text):
    """Lowercase, strip punctuation, and split into words."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


# Common English words that carry no topical meaning
STOP_WORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "of",
    "and", "or", "but", "for", "with", "by", "from", "as", "was",
    "are", "were", "be", "been", "has", "have", "had", "this", "that",
    "these", "those", "its", "their", "they", "also", "which", "into",
    "more", "than", "about", "such", "can", "not", "over", "up", "out",
    "than", "then", "so", "if", "each", "both", "all", "some", "one",
    "two", "three", "four", "five", "six", "seven", "eight", "nine",
    "ten", "other", "new", "after", "before", "between", "through"
}

def meaningful_tokens(text):
    """Return only non-stop-word tokens."""    
    return [t for t in tokenize(text) if t not in STOP_WORDS and len(t) > 2]


# Quick test
sample = all_chunks[0]["text"]
tokens = meaningful_tokens(sample)
print("Sample chunk (first 80 chars):", sample[:80])
print("\nFirst 15 tokens:", tokens[:15])

## 4. Term Frequency (TF)

Term Frequency measures how often a word appears in a single chunk, divided
by the total number of words in that chunk.

```
TF(word, chunk) = count(word in chunk) / total words in chunk
```

A word that appears 5 times in a 100-word chunk has TF = 0.05.

In [ ]:
def term_frequency(tokens):
    """Return a dict of {word: tf_score} for a list of tokens."""
    if not tokens:
        return {}
    counts = Counter(tokens)
    total  = len(tokens)
    return {word: count / total for word, count in counts.items()}


# Demonstrate on the first chunk
sample_tokens = meaningful_tokens(all_chunks[0]["text"])
sample_tf     = term_frequency(sample_tokens)

print("Top 10 terms by TF score in chunk 0:")
top = sorted(sample_tf.items(), key=lambda x: x[1], reverse=True)[:10]
for word, score in top:
    print(f"  {word:20s}  TF = {score:.4f}")

## 5. Inverse Document Frequency (IDF)

IDF rewards rare words. A word that appears in every chunk (like 'world')
tells us very little about which chunk is relevant. A word that appears in
only two chunks is much more informative.

```
IDF(word) = log( total_chunks / chunks_containing_word )
```

In [ ]:
def build_idf(chunks, tokenize_fn):
    """Compute IDF for every word across all chunks."""
    N = len(chunks)
    doc_freq = Counter()
    for chunk in chunks:
        unique_words = set(tokenize_fn(chunk["text"]))
        doc_freq.update(unique_words)

    idf = {}
    for word, df in doc_freq.items():
        idf[word] = math.log(N / df)   # higher when df is small
    return idf


idf_table = build_idf(all_chunks, meaningful_tokens)

print(f"Vocabulary size: {len(idf_table)} unique words\n")

# Words with HIGH IDF appear in very few chunks (very specific)
high_idf = sorted(idf_table.items(), key=lambda x: x[1], reverse=True)[:8]
print("High-IDF words (very specific):")
for w, s in high_idf:
    print(f"  {w:25s}  IDF = {s:.3f}")

print()
# Words with LOW IDF appear in almost every chunk (generic)
low_idf = sorted(idf_table.items(), key=lambda x: x[1])[:8]
print("Low-IDF words (very generic):")
for w, s in low_idf:
    print(f"  {w:25s}  IDF = {s:.3f}")

## 6. TF-IDF Score

The final score multiplies TF by IDF for each query word, then sums across
all query words. Chunks that contain rare, relevant words score highest.

```
TF-IDF(chunk, query) = sum over each query word of  TF(word, chunk) * IDF(word)
```

In [ ]:
def tfidf_score(chunk_tokens, query_tokens, idf_table):
    """Score one chunk against a query using TF-IDF."""    
    tf = term_frequency(chunk_tokens)
    score = 0.0
    for word in query_tokens:
        if word in tf and word in idf_table:
            score += tf[word] * idf_table[word]
    return score


def retrieve(query, chunks, idf_table, tokenize_fn, top_k=3):
    """Return the top-k most relevant chunks for a query."""    
    query_tokens = tokenize_fn(query)
    scored = []
    for chunk in chunks:
        chunk_tokens = tokenize_fn(chunk["text"])
        score = tfidf_score(chunk_tokens, query_tokens, idf_table)
        scored.append((score, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


print("TF-IDF retrieval ready!")

## 7. Run Queries

In [ ]:
def ask(query, top_k=3):
    print(f"Query: {query}")
    print("-" * 55)
    results = retrieve(query, all_chunks, idf_table, meaningful_tokens, top_k)
    for rank, (score, chunk) in enumerate(results, 1):
        preview = chunk["text"][:160].replace("\n", " ")
        print(f"[{rank}] Score: {score:.4f}  Source: {chunk['source']}")
        print(f"    {preview}...")
        print()

ask("What did the Sumerians invent?")

In [ ]:
ask("How does the greenhouse effect work?")

In [ ]:
ask("What is gradient descent in machine learning?")

In [ ]:
ask("Where is the Amazon River located?")

## 8. Key Takeaways

| Concept | Formula | What it measures |
|---------|---------|-----------------|
| TF | count(word) / total_words | How often a word appears in one chunk |
| IDF | log(N / df) | How rare a word is across all chunks |
| TF-IDF | TF x IDF | Combined relevance score |

TF-IDF is fully transparent: you can inspect every calculation.
No embeddings, no neural networks, no GPU required.

**Next notebook:** BM25, a refined version of TF-IDF that handles
document length and term saturation automatically.